# Semantic Model Health · v3 (one-notebook install)

**One notebook does everything:** creates the lakehouse → scans your semantic models (Best Practice Analyzer + VertiPaq + refresh history) → scores each model 0–100 (A–F) → deploys the **Model Health** dashboard (Direct Lake, no refresh or credentials needed) → schedules itself.

### Install: import and Run all
1. Import this notebook into a workspace on Fabric capacity (F2 or above, or Premium/Trial).
2. Click **Run all**. Nothing to edit: the dashboard kit is built into this notebook.
3. Open the *Model Health* report in the same workspace.

Optional: to scan other workspaces than this one, change `WORKSPACES` in CONFIG (e.g. `["Sales", "Finance"]` or `["*"]` for all).

### Built to be gentle on capacity
* Plain Python session (2 vCores), no Spark.
* **Incremental:** models unchanged since the last scan (same last-modified time, no new refresh) are skipped and their results carried forward. A full re-scan is forced every `FULL_SCAN_EVERY_DAYS`.
* One model at a time, a pause between models, a cap per run.
* Direct Lake dashboard: no import refresh, so no refresh load and no stored credentials.
* Detail tables keep only the latest state; history is kept only for compact score tables.

### Scoring (identical in this notebook and in the dashboard DAX)
Per violated rule: `severity weight × min(3, 1 + log10(instances))` with Error = 10, Warning = 4, Info = 1. Log damping means 300 instances of one rule cost about 3× one instance, not 300×. Category score = 100 − points lost (floored at 0). **Health = 30% Performance + 25% DAX + 20% Error Prevention + 15% Maintenance + 10% Formatting.** Grades: A ≥ 90, B ≥ 80, C ≥ 65, D ≥ 50, F < 50.

In [ ]:
%pip install semantic-link-labs --quiet

In [ ]:
# ============================ CONFIG ============================
WORKSPACES          = []      # [] = this workspace | ["Sales", "Finance"] = these | ["*"] = every workspace you can access
EXCLUDE_MODELS      = []                     # exact model names to skip

LAKEHOUSE_NAME      = "LH_ModelHealthAnalyzer"  # created if missing, in this notebook's workspace

# Dashboard (Direct Lake model + report) deployed from the kit
DEPLOY_DASHBOARD    = True
DASHBOARD_NAME      = "Model Health"
# The kit is built into this notebook. Leave these two as they are; they only override it with a newer kit.
KIT_FILE            = "Files/kit/ModelHealthKit.json"   # used only if this file exists in the lakehouse (or a full abfss:// path)
KIT_URL             = None                              # used only if set: raw https URL to a newer ModelHealthKit.json
UPGRADE_DASHBOARD   = False                             # True = overwrite an existing dashboard with the kit

# Schedule this notebook (UTC, HH:MM). None = do not schedule.
SCHEDULE_DAILY_AT   = "05:45"

# Capacity guardrails
FULL_SCAN_EVERY_DAYS = 7      # re-scan unchanged models at least this often
MAX_MODELS_PER_RUN   = 100    # hard cap per run (the rest are picked up next run)
PAUSE_BETWEEN_MODELS = 3      # seconds
RUN_BPA              = True
REFRESH_HISTORY_DEPTH = 20

# Maintainer only: build the kit from an existing Import dashboard in this workspace
BUILD_KIT_FROM_MODEL  = None  # e.g. "Model Health Dashboard"
BUILD_KIT_FROM_REPORT = None  # e.g. "Model Health Dashboard"
PURGE_LEGACY_HISTORY  = False # drop history rows written by v1/v2 (different scoring) before this version
# ================================================================

In [ ]:
import re, math, json, time, base64, datetime, uuid
import pandas as pd, requests
import sempy.fabric as fabric
import sempy_labs as labs
import notebookutils
from deltalake import DeltaTable
import deltalake as _dl

def write_deltalake(*a, **k):
    """Older deltalake needs engine='rust' for schema merges; newer versions removed the argument."""
    if k.get("schema_mode") == "merge":
        try:
            return _dl.write_deltalake(*a, engine="rust", **k)
        except TypeError:
            pass
    return _dl.write_deltalake(*a, **k)

TODAY = datetime.datetime.utcnow().date()
SNAPSHOT_DATE = TODAY.isoformat()
RUN_TS = datetime.datetime.utcnow().replace(microsecond=0).isoformat()
WS_ID = notebookutils.runtime.context["currentWorkspaceId"]
WS_NAME = fabric.resolve_workspace_name(WS_ID)
FABRIC = "https://api.fabric.microsoft.com/v1"

def log(*a): print(*a)

# ---------- Fabric REST with long-running-operation handling ----------
def api(method, path, body=None, ok=(200, 201, 202)):
    h = {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}
    r = requests.request(method, FABRIC + path, headers=h, json=body)
    if r.status_code == 202 and "x-ms-operation-id" in r.headers:
        op = r.headers["x-ms-operation-id"]
        for _ in range(120):
            time.sleep(int(r.headers.get("Retry-After", 3)))
            s = requests.get(f"{FABRIC}/operations/{op}", headers=h).json()
            if s.get("status") in ("Succeeded", "Failed"):
                break
        if s.get("status") != "Succeeded":
            raise RuntimeError(f"{method} {path} failed: {json.dumps(s)[:800]}")
        res = requests.get(f"{FABRIC}/operations/{op}/result", headers=h)
        return res.json() if res.text else {}
    if r.status_code not in ok:
        raise RuntimeError(f"{method} {path} -> HTTP {r.status_code}: {r.text[:800]}")
    return r.json() if r.text else {}

def find_item(ws_id, name, itype):
    items = api("GET", f"/workspaces/{ws_id}/items?type={itype}").get("value", [])
    return next((i for i in items if i["displayName"] == name), None)

# ---------- new items go into the same folder as this notebook ----------
FOLDER_ID = None
try:
    FOLDER_ID = api("GET", f"/workspaces/{WS_ID}/items/{notebookutils.runtime.context.get('currentNotebookId')}").get("folderId")
except Exception:
    pass

def create(path, body, attempts=3):
    """Create an item in this notebook's folder; fall back to the workspace root; retry transient 5xx errors."""
    for i in range(attempts):
        try:
            if FOLDER_ID:
                try:
                    return api("POST", path, {**body, "folderId": FOLDER_ID})
                except RuntimeError as ex:
                    log(f"(folder placement not possible, creating at workspace root: {str(ex)[:80]})")
            return api("POST", path, body)
        except RuntimeError as ex:
            if "HTTP 5" in str(ex) and i < attempts - 1:
                log(f"Transient error, retrying in 20s: {str(ex)[:80]}")
                time.sleep(20)
                existing = find_item(WS_ID, body["displayName"], path.rstrip("s").split("/")[-1].replace("semanticModel", "SemanticModel").replace("lakehouse", "Lakehouse").replace("report", "Report"))
                if existing:
                    return existing
                continue
            raise

# ---------- lakehouse: find or create ----------
lh = find_item(WS_ID, LAKEHOUSE_NAME, "Lakehouse")
if not lh:
    log(f"Creating lakehouse {LAKEHOUSE_NAME} ...")
    lh = create(f"/workspaces/{WS_ID}/lakehouses", {"displayName": LAKEHOUSE_NAME})
    time.sleep(10)
LH_ID = lh["id"]
for _ in range(30):
    props = api("GET", f"/workspaces/{WS_ID}/lakehouses/{LH_ID}").get("properties", {})
    sep = props.get("sqlEndpointProperties") or {}
    if sep.get("provisioningStatus") == "Success":
        break
    time.sleep(10)
SQL_SERVER, SQL_ENDPOINT_ID = sep.get("connectionString"), sep.get("id")
SCHEMA = "dbo/" if props.get("defaultSchema") else ""
ROOT = f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com/{LH_ID}"
TABLES = f"{ROOT}/Tables/{SCHEMA}"

def so():
    return {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}

def read(name):
    try:
        return DeltaTable(TABLES + name, storage_options=so()).to_pandas()
    except Exception:
        return pd.DataFrame()

def sanitize(pdf):
    pdf = pdf.copy(); cols, seen = [], {}
    for c in pdf.columns:
        n = re.sub(r"[^0-9A-Za-z_]", "_", str(c)).strip("_") or "col"
        seen[n] = seen.get(n, -1) + 1
        cols.append(n if seen[n] == 0 else f"{n}_{seen[n]}")
    pdf.columns = cols
    for c in pdf.columns:
        if pd.api.types.is_bool_dtype(pdf[c]) or pd.api.types.is_numeric_dtype(pdf[c]):
            pdf[c] = pd.to_numeric(pdf[c], errors="coerce").astype("float64")
        elif c == "SnapshotDateValue":
            pdf[c] = pd.to_datetime(pdf[c]).dt.date          # Delta 'date' type
        else:
            pdf[c] = pdf[c].astype(str)
    return pdf

def overwrite(pdf, name):
    if pdf is None or not len(pdf):
        return log(f"  [skip] {name}: no rows")
    write_deltalake(TABLES + name, sanitize(pdf).reset_index(drop=True), mode="overwrite", schema_mode="overwrite", storage_options=so())
    log(f"  [ok] {name}: {len(pdf)} rows (replaced)")

def _concat(a, b):
    a = sanitize(a) if a is not None and len(a) else pd.DataFrame()
    b = sanitize(b) if b is not None and len(b) else pd.DataFrame()
    if not len(a): return b
    if not len(b): return a
    for c in set(a.columns) - set(b.columns): b[c] = pd.NA
    for c in set(b.columns) - set(a.columns): a[c] = pd.NA
    out = pd.concat([a, b[a.columns]], ignore_index=True)
    for c in out.columns:  # keep types consistent after aligning columns
        if pd.api.types.is_numeric_dtype(a[c]) or pd.api.types.is_numeric_dtype(b[c]):
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("float64")
        elif c != "SnapshotDateValue":
            out[c] = out[c].astype(str).replace({"<NA>": "", "nan": "", "None": ""})
    return out

def upsert_models(pdf, name, models_to_replace, keep_models):
    """Latest-state tables: replace rows of re-scanned models, drop models that no longer exist,
    stamp everything with today's snapshot. Read-modify-write keeps it independent of Delta writer features."""
    old = read(name)
    if len(old) and "ModelName" in old.columns:
        old = old[~old.ModelName.isin(models_to_replace) & old.ModelName.isin(keep_models)]
    new = _concat(old, pdf)
    if len(new):
        new["SnapshotDate"] = SNAPSHOT_DATE
    overwrite(new, name)

def append_history(pdf, name):
    """History tables: replace today's rows (re-runs on the same day never duplicate)."""
    old = read(name)
    if len(old) and "SnapshotDate" in old.columns:
        old = old[old.SnapshotDate.astype(str) != SNAPSHOT_DATE]
    overwrite(_concat(old, pdf), name)

log(f"Workspace: {WS_NAME} | Lakehouse: {LAKEHOUSE_NAME} | Snapshot: {SNAPSHOT_DATE}")

In [ ]:
# ---------- scoring (identical to the dashboard DAX) ----------
CATEGORY_WEIGHTS = {"Performance": .30, "DAX Expressions": .25, "Error Prevention": .20, "Maintenance": .15, "Formatting": .10}
SEV_WEIGHT = {"Error": 10, "Warning": 4, "Info": 1}
SEV_MAP = {"error": "Error", "❌": "Error", "warning": "Warning", "⚠️": "Warning", "⚠": "Warning",
           "info": "Info", "ℹ️": "Info", "ℹ": "Info"}
norm_sev = lambda v: SEV_MAP.get(str(v).strip().lower(), SEV_MAP.get(str(v).strip(), str(v).strip().title() or "Info"))
grade = lambda s: "A" if s >= 90 else "B" if s >= 80 else "C" if s >= 65 else "D" if s >= 50 else "F"

def score_model(bpa):
    cats = {c: 100.0 for c in CATEGORY_WEIGHTS}
    if bpa is not None and len(bpa):
        g = bpa.groupby(["Category", "Rule_Name", "SeverityNorm"]).size().reset_index(name="n")
        g["pen"] = [SEV_WEIGHT.get(s, 1) * min(3, 1 + math.log10(max(n, 1))) for s, n in zip(g.SeverityNorm, g.n)]
        for cat, pen in g.groupby("Category")["pen"].sum().items():
            if cat in cats:
                cats[cat] = max(0.0, 100.0 - pen)
    return round(sum(cats[c] * w for c, w in CATEGORY_WEIGHTS.items()), 1), cats

RULE_CATALOG = pd.DataFrame([
 ("Column references should be fully qualified", "Write Table[Column] instead of [Column] in measures (Tabular Editor can fix in bulk).", "Low", "Medium"),
 ("Measure references should be unqualified", "Reference measures as [Measure], never Table[Measure].", "Low", "Low"),
 ("Avoid using calculated columns", "Move the logic to Power Query or the lakehouse so it compresses and refreshes faster.", "Medium", "High"),
 ("Reduce usage of calculated tables", "Build these tables in Power Query or the lakehouse instead of DAX.", "Medium", "Medium"),
 ("Remove auto-date table", "Options > Current file > Data load: untick Auto date/time; use one marked Date table.", "Low", "High"),
 ("Model should have a date table", "Add one Date table, mark it as date table, relate it to every fact date.", "Low", "High"),
 ("Date/calendar tables should be marked as a date table", "Table tools > Mark as date table.", "Low", "Medium"),
 ("Do not use floating point data types", "Use Fixed Decimal (currency) or Whole Number instead of Decimal.", "Low", "Medium"),
 ("Set IsAvailableInMdx to false on non-attribute columns", "Set IsAvailableInMdx = false on hidden keys/numeric columns.", "Low", "Medium"),
 ("Check if bi-directional and many-to-many relationships are valid", "Use single-direction filters; use CROSSFILTER only in measures that need it.", "Medium", "High"),
 ("Consider a star-schema instead of a snowflake architecture", "Flatten dimension chains into one dimension per entity.", "High", "High"),
 ("Avoid adding 0 to a measure", "Remove '+ 0'; it forces dense, slow evaluation.", "Low", "Medium"),
 ("Use the DIVIDE function for division", "Replace a / b with DIVIDE(a, b).", "Low", "Low"),
 ("Avoid addition or subtraction of constant values to results of divisions", "Use DIVIDE(a - b, b) instead of DIVIDE(a, b) - 1.", "Low", "Low"),
 ("Measures should not be direct references of other measures", "Delete the duplicate measure or give it distinct logic.", "Low", "Low"),
 ("Inactive relationships that are never activated", "Delete it or use it via USERELATIONSHIP.", "Low", "Low"),
 ("Relationship columns should be of integer data type", "Use whole-number surrogate keys.", "Medium", "Medium"),
 ("Relationship columns should be of the same data type", "Align both key column types in Power Query.", "Low", "High"),
 ("Ensure tables have relationships", "Relate the table or delete it if unused.", "Low", "Low"),
 ("Remove unnecessary columns", "Delete hidden columns not used by measures, relationships or visuals.", "Low", "High"),
 ("Do not summarize numeric columns", "Set Summarize by = None on non-additive numeric columns.", "Low", "Low"),
 ("Whole numbers should be formatted with thousands separators and no decimals", "Format string #,0.", "Low", "Low"),
 ("Percentages should be formatted with thousands separators and 1 decimal", "Format string #,0.0%.", "Low", "Low"),
 ("Provide format string for 'Date' columns", "Set a date format, e.g. dd-mmm-yyyy.", "Low", "Low"),
 ("Provide format string for 'Month' columns", "Set format MMMM yyyy.", "Low", "Low"),
 ("Visible objects with no description", "Add descriptions (they show in tooltips and help Copilot).", "Medium", "Low"),
 ("Hide foreign keys", "Hide relationship key columns on fact tables.", "Low", "Low"),
 ("Mark primary keys", "Set Key column = true on dimension keys.", "Low", "Low"),
], columns=["Rule_Name", "HowToFix", "Effort", "Impact"])

In [ ]:
# ---------- discover models + change detection ----------
target_ws = WORKSPACES or [WS_NAME]
if WORKSPACES == ["*"]:
    wsl = fabric.list_workspaces()
    target_ws = [w for w in wsl["Name"].tolist() if w]

prev_snap = read("model_snapshots")
prev_latest = {}
if len(prev_snap) and "Signature" in prev_snap.columns:
    p = prev_snap.sort_values("SnapshotDate").groupby("ModelName").tail(1)
    prev_latest = {r.ModelName: r for r in p.itertuples()}

skip = set(EXCLUDE_MODELS) | {DASHBOARD_NAME, BUILD_KIT_FROM_MODEL or ""}
found, inventory, name_count = [], [], {}
for ws in target_ws:
    try:
        ds = fabric.list_datasets(workspace=ws)
    except Exception as ex:
        log(f"[warn] cannot read '{ws}': {ex}"); continue
    ncol = "Dataset Name" if "Dataset Name" in ds.columns else ds.columns[0]
    ucol = next((c for c in ds.columns if "update" in c.lower()), None)
    for _, row in ds.iterrows():
        m = row[ncol]
        if m in skip or m == LAKEHOUSE_NAME:
            continue
        found.append({"ws": ws, "model": m, "last_update": str(row[ucol]) if ucol else ""})
        name_count[m] = name_count.get(m, 0) + 1
    inv = ds.copy(); inv["WorkspaceName"] = ws; inventory.append(inv)

for f in found:
    f["name"] = f["model"] if name_count[f["model"]] == 1 else f"{f['model']} [{f['ws']}]"

log(f"{len(found)} model(s) in {len(target_ws)} workspace(s)")

In [ ]:
# ---------- analyze (incremental, one model at a time) ----------
out = {k: [] for k in ["tables", "columns", "measures", "rels", "bpa", "refresh"]}
snapshots, scores, rule_hist, scanned, carried = [], [], [], [], []
budget = MAX_MODELS_PER_RUN

for f in found:
    ws, model, name = f["ws"], f["model"], f["name"]
    # cheap signature: last metadata update + latest refresh
    rh = pd.DataFrame()
    try:
        rh = fabric.list_refresh_requests(dataset=model, workspace=ws, top_n=REFRESH_HISTORY_DEPTH)
    except Exception:
        pass
    end_col = next((c for c in rh.columns if "end" in c.lower()), None) if len(rh) else None
    last_refresh = str(rh[end_col].max()) if end_col else ""
    signature = f"{f['last_update']}|{last_refresh}"

    prev = prev_latest.get(name)
    days_since_full = (TODAY - datetime.date.fromisoformat(str(prev.LastFullScan)[:10])).days if prev is not None and getattr(prev, "LastFullScan", None) not in (None, "", "nan") else 999
    unchanged = prev is not None and getattr(prev, "Signature", None) == signature and getattr(prev, "AnalysisStatus", "") == "OK"
    if (unchanged and days_since_full < FULL_SCAN_EVERY_DAYS) or budget <= 0:
        carried.append(name)
        log(f"= {name}: unchanged, carried forward" if budget > 0 else f"= {name}: run cap reached, carried forward")
        continue

    budget -= 1
    t0 = time.time()
    log(f"\n=== {name} ===")
    snap = {"ModelName": name, "WorkspaceName": ws, "SnapshotDate": SNAPSHOT_DATE, "RunTimestamp": RUN_TS,
            "AnalysisStatus": "OK", "ErrorDetail": "", "Signature": signature, "LastFullScan": SNAPSHOT_DATE, "ScanType": "full"}
    bpa = None
    ctx = lambda d: d.assign(ModelName=name, WorkspaceName=ws, SnapshotDate=SNAPSHOT_DATE, RunTimestamp=RUN_TS)
    try:
        try:
            t = fabric.list_tables(dataset=model, workspace=ws, extended=True)
        except Exception:
            t = fabric.list_tables(dataset=model, workspace=ws)
        try:
            c = fabric.list_columns(dataset=model, workspace=ws, extended=True)
        except Exception:
            c = fabric.list_columns(dataset=model, workspace=ws)
        m = fabric.list_measures(dataset=model, workspace=ws)
        try:
            r = fabric.list_relationships(dataset=model, workspace=ws)
        except Exception:
            r = pd.DataFrame()
        size_col = next((x for x in c.columns if str(x).lower().replace(" ", "") == "totalsize"), None)
        size_mb = float(pd.to_numeric(c[size_col], errors="coerce").fillna(0).sum()) / 1048576 if size_col else 0.0
        if RUN_BPA:
            try:
                bpa = labs.run_model_bpa(dataset=model, workspace=ws, return_dataframe=True)
                if bpa is not None and len(bpa):
                    bpa.columns = [re.sub(r"[^0-9A-Za-z_]", "_", str(x)).strip("_") for x in bpa.columns]
                    sev = next((x for x in bpa.columns if "sever" in x.lower()), None)
                    bpa["SeverityNorm"] = bpa[sev].apply(norm_sev) if sev else "Info"
            except Exception as ex:
                log(f"  BPA failed: {ex}")
        if len(rh):
            pick = lambda *keys: next((x for x in rh.columns if all(k in x.lower() for k in keys)), None)
            st, en, ty, stc, err = pick("start"), pick("end"), pick("type"), pick("status"), pick("exception") or pick("error")
            norm = pd.DataFrame({"StartTime": rh[st].astype(str) if st else "", "EndTime": rh[en].astype(str) if en else "",
                                 "RefreshType": rh[ty].astype(str) if ty else "", "Status": rh[stc].astype(str) if stc else "",
                                 "Error": rh[err].astype(str).str[:300] if err else ""})
            if st and en:
                norm["DurationSec"] = (pd.to_datetime(rh[en], errors="coerce", utc=True) - pd.to_datetime(rh[st], errors="coerce", utc=True)).dt.total_seconds()
            rh = norm
        for k, d in (("tables", t), ("columns", c), ("measures", m), ("rels", r), ("bpa", bpa), ("refresh", rh)):
            if d is not None and len(d):
                out[k].append(ctx(d))
        health, cats = score_model(bpa)
        sev = bpa["SeverityNorm"].value_counts().to_dict() if bpa is not None and len(bpa) else {}
        snap.update(TableCount=len(t), ColumnCount=len(c), MeasureCount=len(m), RelationshipCount=len(r),
                    ModelSizeMB=round(size_mb, 2), DurationSec=round(time.time() - t0, 1))
        scores.append({"ModelName": name, "WorkspaceName": ws, "SnapshotDate": SNAPSHOT_DATE, "RunTimestamp": RUN_TS,
                       "HealthScore": health, "Grade": grade(health), "ScanType": "full",
                       "ScorePerformance": round(cats["Performance"], 1), "ScoreDAXExpressions": round(cats["DAX Expressions"], 1),
                       "ScoreErrorPrevention": round(cats["Error Prevention"], 1), "ScoreMaintenance": round(cats["Maintenance"], 1),
                       "ScoreFormatting": round(cats["Formatting"], 1), "ErrorCount": sev.get("Error", 0),
                       "WarningCount": sev.get("Warning", 0), "InfoCount": sev.get("Info", 0),
                       "TotalViolations": sum(sev.values()), "ModelSizeMB": round(size_mb, 2)})
        if bpa is not None and len(bpa):
            rule_hist.append(ctx(bpa.groupby(["Category", "Rule_Name", "SeverityNorm"]).size().reset_index(name="Instances")))
        scanned.append(name)
        log(f"  {health} ({grade(health)}) | {sum(sev.values())} violations | {size_mb:.1f} MB | {snap['DurationSec']}s")
    except Exception as ex:
        snap.update(AnalysisStatus="FAILED", ErrorDetail=str(ex)[:500]); scanned.append(name)
        log(f"  FAILED: {ex}")
    snapshots.append(snap)
    time.sleep(PAUSE_BETWEEN_MODELS)

log(f"\nScanned {len(scanned)}, carried forward {len(carried)}")

In [ ]:
# ---------- write ----------
cat = lambda fr: pd.concat(fr, ignore_index=True) if fr else pd.DataFrame()
current = [f["name"] for f in found]

def carry(hist_name, new_rows):
    """Copy the latest history rows of carried-forward models to today's snapshot."""
    h = read(hist_name)
    if not len(h) or not carried:
        return new_rows
    h = h[h.ModelName.isin(carried)]
    if not len(h):
        return new_rows
    last = h.groupby("ModelName")["SnapshotDate"].transform("max")
    h = h[h.SnapshotDate == last].copy()
    h["SnapshotDate"] = SNAPSHOT_DATE; h["RunTimestamp"] = RUN_TS
    if "ScanType" in h.columns:
        h["ScanType"] = "carried"
    return pd.concat([new_rows, h], ignore_index=True)

log("Writing:")
if PURGE_LEGACY_HISTORY:
    for hname in ("health_scores", "model_snapshots", "bpa_rule_history"):
        h = read(hname)
        if len(h):
            keep = h[h.SnapshotDate.astype(str) >= SNAPSHOT_DATE] if "ScanType" not in h.columns else h[h.ScanType.astype(str).isin(["full", "carried"])]
            overwrite(keep if len(keep) else h.head(0), hname)
append_history(carry("health_scores", pd.DataFrame(scores)), "health_scores")
append_history(carry("model_snapshots", pd.DataFrame(snapshots)), "model_snapshots")
append_history(carry("bpa_rule_history", cat(rule_hist)), "bpa_rule_history")
for key, tbl in (("bpa", "bpa_violations"), ("tables", "table_stats"), ("columns", "column_stats"),
                 ("measures", "measure_inventory"), ("rels", "relationship_stats")):
    upsert_models(cat(out[key]), tbl, scanned, keep_models=current)

# refresh history always exists with a fixed schema, so the dashboard never misses the table
REFRESH_COLS = ["ModelName", "WorkspaceName", "SnapshotDate", "RunTimestamp", "StartTime", "EndTime", "RefreshType", "Status", "Error"]
rf = cat(out["refresh"])
if not len(rf) and not len(read("refresh_history")):
    rf = pd.DataFrame([{**{c: "" for c in REFRESH_COLS}, "ModelName": n, "WorkspaceName": "", "Status": "No refresh history", "DurationSec": 0.0} for n in current[:1] or ["(none)"]])
upsert_models(rf, "refresh_history", scanned, keep_models=current + (["(none)"] if not current else []))

# dimensions are built from EVERY table, so no fact row can ever be orphaned (no blank rows in the report)
FACTS = ["health_scores", "model_snapshots", "bpa_rule_history", "bpa_violations", "table_stats", "column_stats",
         "measure_inventory", "relationship_stats", "refresh_history"]
names, dates = set(), set()
for t in FACTS:
    d = read(t)
    if len(d) and "ModelName" in d.columns:
        names |= set(d.ModelName.astype(str))
    if len(d) and "SnapshotDate" in d.columns:
        dates |= set(d.SnapshotDate.astype(str))
names.discard(""); dates.discard("")
hs = read("health_scores")
latest = hs[hs.SnapshotDate.astype(str) == SNAPSHOT_DATE][["ModelName", "WorkspaceName", "HealthScore", "Grade"]]
dm = pd.DataFrame({"ModelName": sorted(names)}).merge(latest, on="ModelName", how="left")
overwrite(dm.rename(columns={"HealthScore": "LatestScore", "Grade": "LatestGrade"}), "dim_models")
snaps = pd.DataFrame({"SnapshotKey": sorted(dates)})
snaps["SnapshotDateValue"] = pd.to_datetime(snaps.SnapshotKey)
overwrite(snaps, "dim_snapshots")
inv = cat(inventory)
overwrite(inv.assign(SnapshotDate=SNAPSHOT_DATE) if len(inv) else inv, "model_inventory")
overwrite(RULE_CATALOG, "rule_catalog")

try:
    api("POST", f"/workspaces/{WS_ID}/sqlEndpoints/{SQL_ENDPOINT_ID}/refreshMetadata", {})
    log("SQL endpoint metadata synced")
except Exception as ex:
    log(f"SQL endpoint sync skipped: {ex}")

In [ ]:
# ---------- maintainer: build ModelHealthKit.json from an Import dashboard ----------
TYPE_MAP = {"string": "string", "double": "double", "float": "double", "long": "int64", "integer": "int64",
            "short": "int64", "byte": "int64", "boolean": "boolean", "date": "dateTime", "timestamp": "dateTime",
            "timestamp_ntz": "dateTime"}
TABLE_SOURCES = {"bpa_violations": "bpa_violations", "column_stats": "column_stats", "health_scores": "health_scores",
                 "measure_inventory": "measure_inventory", "model_snapshots": "model_snapshots",
                 "relationship_stats": "relationship_stats", "table_stats": "table_stats",
                 "Models": "dim_models", "Snapshots": "dim_snapshots", "bpa_rule_history": "bpa_rule_history",
                 "refresh_history": "refresh_history", "model_inventory": "model_inventory", "rule_catalog": "rule_catalog"}
RENAME = {"Models": {"ModelName": "Model Name"}, "Snapshots": {"SnapshotKey": "Snapshot Key", "SnapshotDateValue": "Snapshot Date"}}
NEW_RELS = [("bpa_rule_history", "ModelName", "Models", "Model Name"), ("bpa_rule_history", "SnapshotDate", "Snapshots", "Snapshot Key"),
            ("refresh_history", "ModelName", "Models", "Model Name"), ("bpa_violations", "Rule_Name", "rule_catalog", "Rule_Name")]
NEW_MEASURES = [
 ("Score Change vs Previous", """VAR s = CALCULATE ( MAX ( 'Snapshots'[Snapshot Key] ), ALL ( 'Snapshots' ) )
VAR p = CALCULATE ( MAX ( 'Snapshots'[Snapshot Key] ), ALL ( 'Snapshots' ), 'Snapshots'[Snapshot Key] < s )
VAR cur = CALCULATE ( AVERAGE ( health_scores[HealthScore] ), ALL ( 'Snapshots' ), 'Snapshots'[Snapshot Key] = s )
VAR prv = CALCULATE ( AVERAGE ( health_scores[HealthScore] ), ALL ( 'Snapshots' ), 'Snapshots'[Snapshot Key] = p )
RETURN IF ( NOT ISBLANK ( p ), cur - prv )""", "+0.0;-0.0;0.0", "01 Health Score", "Change in health score versus the previous scan (from score history)."),
 ("Health Score (History)", "AVERAGE ( health_scores[HealthScore] )", "0.0", "05 Trend", "Score per scan date, for trend lines over 'Snapshots'[Snapshot Date]."),
 ("Last Scan", "CALCULATE ( MAX ( 'Snapshots'[Snapshot Date] ), ALL ( 'Snapshots' ) )", "dd mmm yyyy", "05 Trend", "Date of the most recent scan."),
 ("Fix Guidance", "VAR r = SELECTEDVALUE ( bpa_violations[Rule_Name] ) RETURN COALESCE ( LOOKUPVALUE ( rule_catalog[HowToFix], rule_catalog[Rule_Name], r ), \"See the rule description\" )", "", "02 Findings", "How to fix the rule in the current context (from rule_catalog)."),
 ("Fix Effort", "VAR r = SELECTEDVALUE ( bpa_violations[Rule_Name] ) RETURN LOOKUPVALUE ( rule_catalog[Effort], rule_catalog[Rule_Name], r )", "", "02 Findings", "Low / Medium / High effort to fix the rule in context."),
 ("Fix Impact", "VAR r = SELECTEDVALUE ( bpa_violations[Rule_Name] ) RETURN LOOKUPVALUE ( rule_catalog[Impact], rule_catalog[Rule_Name], r )", "", "02 Findings", "Low / Medium / High impact of fixing the rule in context."),
 ("Refresh Failures", "CALCULATE ( COUNTROWS ( refresh_history ), refresh_history[Status] = \"Failed\" )", "#,0", "06 Refresh", "Failed refreshes among the most recent refreshes per model."),
 ("Refresh Success Rate", "DIVIDE ( CALCULATE ( COUNTROWS ( refresh_history ), refresh_history[Status] = \"Completed\" ), CALCULATE ( COUNTROWS ( refresh_history ), refresh_history[Status] IN { \"Completed\", \"Failed\" } ) )", "0.0%", "06 Refresh", "Share of recent refreshes that completed."),
]

def delta_fields(lh_table):
    try:
        return [(f.name, str(f.type.type) if hasattr(f.type, "type") else str(f.type)) for f in DeltaTable(TABLES + lh_table, storage_options=so()).schema().fields]
    except Exception:
        return []

def build_kit(model_name, report_name):
    src_model = find_item(WS_ID, model_name, "SemanticModel")
    src_report = find_item(WS_ID, report_name, "Report")
    bim = labs.get_semantic_model_bim(dataset=model_name, workspace=WS_NAME)
    mdl = bim["model"]
    old = {t["name"]: t for t in mdl.get("tables", [])}
    tables = []
    for tname, lh_table in TABLE_SOURCES.items():
        fields = delta_fields(lh_table)
        if not fields:
            log(f"  [kit] skip {tname}: no lakehouse table {lh_table}"); continue
        t = old.get(tname, {"name": tname})
        t = {k: v for k, v in t.items() if k not in ("partitions", "columns")}
        oldcols = {c["name"]: c for c in old.get(tname, {}).get("columns", []) if c.get("type") not in ("calculated", "rowNumber", "calculatedTableColumn")}
        cols = []
        for fname, ftype in fields:
            if fname.lower().lstrip("_").startswith("index_level"):
                continue
            cname = RENAME.get(tname, {}).get(fname, fname)
            c = {k: v for k, v in oldcols.get(cname, {}).items() if k not in ("type", "expression", "sourceColumn", "dataType", "isNameInferred", "isDataTypeInferred", "sourceProviderType")}
            c.update({"name": cname, "sourceColumn": fname, "dataType": TYPE_MAP.get(ftype, "string")})
            if c["dataType"] in ("double", "int64"):
                c.setdefault("summarizeBy", "none")
            cols.append(c)
        t["columns"] = cols
        t["partitions"] = [{"name": tname, "mode": "directLake",
                            "source": {"type": "entity", "entityName": lh_table, "schemaName": "dbo", "expressionSource": "DatabaseQuery"}}]
        tables.append(t)
    names = {t["name"]: {c["name"] for c in t["columns"]} for t in tables}
    rels = [r for r in mdl.get("relationships", []) if r["fromTable"] in names and r["toTable"] in names
            and r["fromColumn"] in names[r["fromTable"]] and r["toColumn"] in names[r["toTable"]]]
    have = {(r["fromTable"], r["fromColumn"], r["toTable"]) for r in rels}
    for ft, fc, tt, tc in NEW_RELS:
        if ft in names and tt in names and fc in names[ft] and tc in names[tt] and (ft, fc, tt) not in have:
            rels.append({"name": str(uuid.uuid4()), "fromTable": ft, "fromColumn": fc, "toTable": tt, "toColumn": tc})
    home = next(t for t in tables if t["name"] == "health_scores")
    home.setdefault("measures", [])
    for n, e, fmt, folder, desc in NEW_MEASURES:
        if "refresh_history" in e and "refresh_history" not in names:
            continue
        if "refresh_history[Status]" in e and "Status" not in names.get("refresh_history", set()):
            continue
        for tbl in tables:
            tbl["measures"] = [x for x in tbl.get("measures", []) if x["name"] != n]
        mm = {"name": n, "expression": e, "displayFolder": folder, "description": desc}
        if fmt:
            mm["formatString"] = fmt
        home["measures"].append(mm)
    snap_tbl = next((t for t in tables if t["name"] == "Snapshots"), None)
    if snap_tbl:
        snap_tbl["dataCategory"] = "Time"
        for c in snap_tbl["columns"]:
            if c["name"] == "Snapshot Date":
                c["isKey"] = True; c["formatString"] = "dd mmm yyyy"
    mdl["tables"] = tables
    mdl["relationships"] = rels
    mdl["expressions"] = [{"name": "DatabaseQuery", "kind": "m",
                           "expression": ["let", '    database = Sql.Database("{{SQL_SERVER}}", "{{SQL_ENDPOINT_ID}}")', "in", "    database"]}]
    for k in ("dataSources", "queryGroups"):
        mdl.pop(k, None)
    mdl["annotations"] = [a for a in mdl.get("annotations", []) if a.get("name") != "PBI_QueryOrder"]
    bim["compatibilityLevel"] = max(1604, bim.get("compatibilityLevel", 1604))
    bim["name"] = "{{MODEL_NAME}}"
    rep = api("POST", f"/workspaces/{WS_ID}/reports/{src_report['id']}/getDefinition")
    parts = rep["definition"]["parts"]

    def latest_dates(node):
        """Date fields aggregated with Min/Earliest -> Max/Latest (snapshot cards)."""
        n = 0
        if isinstance(node, dict):
            agg = node.get("Aggregation")
            if isinstance(agg, dict) and agg.get("Function") == 3:
                col = (agg.get("Expression") or {}).get("Column") or {}
                if col.get("Property") in ("Snapshot Date", "Snapshot Key", "SnapshotDate"):
                    agg["Function"] = 4; n += 1
            for k, v in list(node.items()):
                if isinstance(v, str) and ("Min(Snapshots." in v or v.startswith("Earliest Snapshot")):
                    node[k] = v.replace("Min(Snapshots.", "Max(Snapshots.").replace("Earliest Snapshot", "Latest Snapshot"); n += 1
                else:
                    n += latest_dates(v)
        elif isinstance(node, list):
            n += sum(latest_dates(x) for x in node)
        return n

    patched = 0
    for p in parts:
        if p["path"].endswith(".json"):
            try:
                doc = json.loads(base64.b64decode(p["payload"]).decode("utf-8"))
                k = latest_dates(doc)
                if k:
                    p["payload"] = base64.b64encode(json.dumps(doc).encode("utf-8")).decode(); patched += k
            except Exception:
                pass
    log(f"[kit] date fields switched to latest snapshot: {patched}")
    for p in parts:
        if p["path"] == "definition.pbir":
            txt = base64.b64decode(p["payload"]).decode("utf-8")
            txt = txt.replace(src_model["id"], "{{MODEL_ID}}").replace(WS_ID, "{{WORKSPACE_ID}}").replace(WS_NAME, "{{WORKSPACE_NAME}}").replace(model_name, "{{MODEL_NAME}}")
            p["payload"] = base64.b64encode(txt.encode("utf-8")).decode()
    kit = {"kit_version": "3.0", "built": RUN_TS, "model_bim": bim, "report_parts": parts,
           "report_format": rep["definition"].get("format")}
    notebookutils.fs.put(f"{ROOT}/{KIT_FILE}", json.dumps(kit), True)
    log(f"[kit] wrote {KIT_FILE}: {len(tables)} tables, {len(rels)} relationships, {sum(len(t.get('measures', [])) for t in tables)} measures, {len(parts)} report parts")
    return kit

KIT = None
if BUILD_KIT_FROM_MODEL:
    KIT = build_kit(BUILD_KIT_FROM_MODEL, BUILD_KIT_FROM_REPORT or BUILD_KIT_FROM_MODEL)

### Built-in dashboard kit (do not edit)
The Direct Lake model and report definition, compressed and split over a few cells (Fabric limits a cell to 500k characters). These cells only store data.

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 1 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED = (
    "..."
)

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 2 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED += (
    "..."
)

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 3 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED += (
    "..."
)

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 4 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED += (
    "..."
)

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 5 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED += (
    "..."
)

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 6 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED += (
    "..."
)

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 7 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED += (
    "..."
)

In [ ]:
# Built-in Model Health dashboard kit v3.0, part 8 of 8 (gzip + base64). Do not edit.
KIT_EMBEDDED += (
    "..."
)

In [ ]:
# ---------- deploy the dashboard (Direct Lake model + report) from the kit ----------
b64 = lambda s: base64.b64encode(s.encode("utf-8")).decode()

def load_kit():
    if KIT is not None:
        return KIT
    if KIT_URL:
        return requests.get(KIT_URL, timeout=60).json()
    try:
        path = KIT_FILE if KIT_FILE.startswith("abfss://") else f"{ROOT}/{KIT_FILE}"
        if notebookutils.fs.exists(path):
            log(f"[kit] using {KIT_FILE}")
            return json.loads(notebookutils.fs.head(path, 50_000_000))
    except Exception:
        pass
    import gzip
    log("[kit] using the kit built into this notebook")
    return json.loads(gzip.decompress(base64.b64decode(KIT_EMBEDDED)).decode("utf-8"))

def prepare_kit(kit):
    """Make a kit environment-neutral: no saved slicer selections, no blank model rows."""
    bim = json.loads(json.dumps(kit["model_bim"]))
    for t in bim["model"]["tables"]:
        for m in t.get("measures", []):
            e = m["expression"] if isinstance(m["expression"], str) else "\n".join(m["expression"])
            e = (e.replace("ALL ( 'Models'[Model Name] )", "ALLNOBLANKROW ( 'Models'[Model Name] )")
                  .replace("ALL('Models'[Model Name])", "ALLNOBLANKROW('Models'[Model Name])")
                  .replace("VALUES ( 'Models'[Model Name] )", "FILTER ( VALUES ( 'Models'[Model Name] ), NOT ISBLANK ( 'Models'[Model Name] ) )"))
            cat = {"Score Performance": "Performance", "Score DAX Expressions": "DAX Expressions", "Score Error Prevention": "Error Prevention",
                   "Score Maintenance": "Maintenance", "Score Formatting": "Formatting"}.get(m["name"])
            if cat and "AVERAGEX" not in e:   # several models selected = average of the model scores
                e = ("AVERAGEX ( FILTER ( VALUES ( 'Models'[Model Name] ), NOT ISBLANK ( 'Models'[Model Name] ) ), "
                     f"VAR p = CALCULATE ( [Category Penalty], 'bpa_violations'[Category] = \"{cat}\" ) RETURN MAX ( 0, 100 - COALESCE ( p, 0 ) ) )")
            m["expression"] = e
    not_blank = lambda entity, prop: {"name": f"nb_{uuid.uuid4().hex[:16]}", "type": "Categorical",
        "field": {"Column": {"Expression": {"SourceRef": {"Entity": entity}}, "Property": prop}},
        "filter": {"Version": 2, "From": [{"Name": "m", "Entity": entity, "Type": 0}],
                   "Where": [{"Condition": {"Not": {"Expression": {"In": {
                       "Expressions": [{"Column": {"Expression": {"SourceRef": {"Source": "m"}}, "Property": prop}}],
                       "Values": [[{"Literal": {"Value": "null"}}]]}}}}}]}}
    # keep only columns that exist in this lakehouse, so Direct Lake can always load the model
    for t in bim["model"]["tables"]:
        part = (t.get("partitions") or [{}])[0].get("source", {})
        fields = {f for f, _ in delta_fields(part.get("entityName", ""))} if part.get("entityName") else set()
        if fields:
            gone = [c["name"] for c in t.get("columns", []) if c.get("sourceColumn") and c["sourceColumn"] not in fields]
            if gone:
                t["columns"] = [c for c in t["columns"] if c["name"] not in gone]
                log(f"[kit] {t['name']}: skipped columns not in the lakehouse: {', '.join(gone)}")
    parts = []
    for p in kit["report_parts"]:
        p = dict(p)
        if p["path"].endswith("visual.json"):
            try:
                v = json.loads(base64.b64decode(p["payload"]).decode("utf-8"))
                vis = v.get("visual", {})
                if vis.get("visualType", "").lower().startswith("slicer") and '"Models"' in json.dumps(vis):
                    for g in (vis.get("objects", {}) or {}).get("general", []) or []:
                        (g.get("properties") or {}).pop("filter", None)
                    fc = v.setdefault("filterConfig", {}).setdefault("filters", [])
                    fc.append(not_blank("Models", "Model Name"))
                    p["payload"] = base64.b64encode(json.dumps(v).encode("utf-8")).decode()
            except Exception as ex:
                log(f"(kit visual skipped: {ex})")
        parts.append(p)
    return {**kit, "model_bim": bim, "report_parts": parts}

if DEPLOY_DASHBOARD:
    kit = load_kit()
    kit = prepare_kit(kit) if kit else kit
    if not kit:
        log(f"No kit found at {KIT_FILE} (or KIT_URL). Upload ModelHealthKit.json to deploy the dashboard; data is ready.")
    else:
        bim_txt = (json.dumps(kit["model_bim"]).replace("{{SQL_SERVER}}", SQL_SERVER)
                   .replace("{{SQL_ENDPOINT_ID}}", SQL_ENDPOINT_ID).replace("{{MODEL_NAME}}", DASHBOARD_NAME))
        model_def = {"parts": [{"path": "model.bim", "payload": b64(bim_txt), "payloadType": "InlineBase64"},
                               {"path": "definition.pbism", "payload": b64(json.dumps({"version": "1.0"})), "payloadType": "InlineBase64"}]}
        sm = find_item(WS_ID, DASHBOARD_NAME, "SemanticModel")
        if sm and UPGRADE_DASHBOARD:
            api("POST", f"/workspaces/{WS_ID}/semanticModels/{sm['id']}/updateDefinition", {"definition": model_def})
            log(f"Dashboard model upgraded: {DASHBOARD_NAME}")
        elif not sm:
            create(f"/workspaces/{WS_ID}/semanticModels", {"displayName": DASHBOARD_NAME, "definition": model_def})
            sm = find_item(WS_ID, DASHBOARD_NAME, "SemanticModel")
            log(f"Dashboard model created: {DASHBOARD_NAME} (Direct Lake)")
        try:
            fabric.refresh_dataset(dataset=DASHBOARD_NAME, workspace=WS_NAME, refresh_type="full")
        except Exception as ex:
            log(f"Reframe skipped: {ex}")

        parts = []
        for p in kit["report_parts"]:
            payload = p["payload"]
            if p["path"] == "definition.pbir":
                txt = base64.b64decode(payload).decode("utf-8")
                txt = (txt.replace("{{MODEL_ID}}", sm["id"]).replace("{{WORKSPACE_ID}}", WS_ID)
                          .replace("{{WORKSPACE_NAME}}", WS_NAME).replace("{{MODEL_NAME}}", DASHBOARD_NAME))
                payload = b64(txt)
            parts.append({"path": p["path"], "payload": payload, "payloadType": "InlineBase64"})
        rep_def = {"parts": parts}
        if kit.get("report_format"):
            rep_def["format"] = kit["report_format"]
        rp = find_item(WS_ID, DASHBOARD_NAME, "Report")
        if rp and UPGRADE_DASHBOARD:
            api("POST", f"/workspaces/{WS_ID}/reports/{rp['id']}/updateDefinition", {"definition": rep_def})
            log("Dashboard report upgraded")
        elif not rp:
            create(f"/workspaces/{WS_ID}/reports", {"displayName": DASHBOARD_NAME, "definition": rep_def})
            log(f"Dashboard report created: {DASHBOARD_NAME}")
        else:
            log("Dashboard already installed (set UPGRADE_DASHBOARD = True to update it)")

In [ ]:
# ---------- schedule this notebook ----------
if SCHEDULE_DAILY_AT:
    try:
        nb_id = notebookutils.runtime.context.get("currentNotebookId")
        path = f"/workspaces/{WS_ID}/items/{nb_id}/jobs/RunNotebook/schedules"
        existing = api("GET", path).get("value", [])
        if existing:
            log(f"Schedule already exists ({len(existing)}); leaving it as is")
        else:
            start = datetime.datetime.utcnow().replace(microsecond=0)
            api("POST", path, {"enabled": True, "configuration": {
                "startDateTime": start.isoformat() + "Z", "endDateTime": (start + datetime.timedelta(days=3650)).isoformat() + "Z",
                "localTimeZoneId": "UTC", "type": "Daily", "times": [SCHEDULE_DAILY_AT]}})
            log(f"Scheduled daily at {SCHEDULE_DAILY_AT} UTC")
    except Exception as ex:
        log(f"Scheduling skipped ({ex}). Use Run > Schedule in the notebook toolbar instead.")

In [ ]:
# ---------- summary ----------
hs = read("health_scores")
now = hs[hs.SnapshotDate == SNAPSHOT_DATE].sort_values("HealthScore")
print(now[[c for c in ["ModelName", "WorkspaceName", "HealthScore", "Grade", "TotalViolations", "ModelSizeMB", "ScanType"] if c in now.columns]].to_string(index=False))
print(f"\nScanned {len(scanned)} | carried forward {len(carried)} | failed {sum(1 for s in snapshots if s['AnalysisStatus'] != 'OK')}")

# self-check: every fact row must match a model and a snapshot in the dimensions
dm_names = set(read("dim_models").ModelName.astype(str))
dm_dates = set(read("dim_snapshots").SnapshotKey.astype(str))
issues = []
for t in FACTS:
    d = read(t)
    if len(d):
        o1 = int((~d.ModelName.astype(str).isin(dm_names)).sum()) if "ModelName" in d.columns else 0
        o2 = int((~d.SnapshotDate.astype(str).isin(dm_dates)).sum()) if "SnapshotDate" in d.columns else 0
        if o1 or o2:
            issues.append(f"{t}: {o1} rows without model, {o2} rows without snapshot")
print("Self-check:", "OK" if not issues else "; ".join(issues))